In [1]:
import pandas as pd
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, text
from datetime import datetime
from dateutil.relativedelta import relativedelta

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [23]:
class Updater:
    def __init__(self):
        
        self.__get_engine()
        
    def __load_dotenv(self):
        load_dotenv()
        return {
            'user': os.getenv("DB_USER"),
            'password': os.getenv("DB_PASSWORD"),
            'host': '172.16.3.158',
            'port': os.getenv("DB_PORT"),
            'db_name': os.getenv("DB_NAME"),
            'schema': 'fin'
        }

    def __get_engine(self):
        conf = self.__load_dotenv()
        self.__engine = create_engine(f"postgresql+psycopg2://{conf['user']}:{conf['password']}@{conf['host']}:{conf['port']}/{conf['db_name']}")

    def __read_sql_query(self, query):
        with self.__engine.begin() as con:
            return pd.read_sql_query(query, con)

    def pandas_to_db(self, df, table, exists):
        with self.__engine.connect() as con:
            df.to_sql(table, con, if_exists=exists, index=False, schema='fin')

In [7]:
df = pd.read_csv("revenue_plan_2026.csv", sep=';')[['company', 'date_dt', 'frc', 'amount']]

In [15]:
df = (
    df
    .assign(date_dt = lambda x: pd.to_datetime(x['date_dt'], dayfirst=True))
    .assign(amount = lambda x: x['amount'].str.replace(',', '.').astype('float64'))
)

In [24]:
upd = Updater()
upd.pandas_to_db(df, 'revenue_plan_2025', 'append')